# Глава 30. Временные ряды: прогноз во времени

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, чем задачи с временными рядами отличаются от обычных, правильно разделить такие данные (по времени, не случайно) и построить простой прогноз со скользящим средним как baseline.

Прогноз продаж на следующий месяц, курса валют на завтра, погоды на неделю — всё это временные ряды: данные, где важен порядок во времени. Это один из самых практичных типов задач, и у него есть свои строгие правила, нарушение которых незаметно всё портит.

**Аналогия: нельзя подсматривать в будущее  💡**

Прогнозировать временной ряд — как предсказывать завтрашнюю погоду, зная только сегодняшнюю и вчерашнюю. Ты не имеешь права заглянуть в послезавтра, чтобы «потренироваться». В обычных задачах мы случайно перемешивали данные для train/test. Здесь так делать НЕЛЬЗЯ: перемешав, ты дашь модели подсмотреть будущее при обучении, и оценка станет ложной. Время течёт только вперёд — и разбивать данные надо тоже по времени.

## Главное правило: разбивай по времени

Это ключевое отличие временных рядов. Обучающая выборка — прошлое, тестовая — будущее относительно неё. Никакого random split, иначе получится утечка данных из будущего.

In [ ]:
# === Подготовка: синтетический временной ряд (дата + значение) ===
import pandas as pd, numpy as np
dates = pd.date_range("2023-01-01", periods=200, freq="D")
trend = np.linspace(10, 30, 200)
seasonality = 5 * np.sin(np.arange(200) * 2 * np.pi / 7)
noise = np.random.RandomState(42).normal(0, 2, 200)
df = pd.DataFrame({"date": dates, "value": trend + seasonality + noise})

import pandas as pd

# Данные упорядочены по дате
df = df.sort_values("date")

# Разбиваем ПО ВРЕМЕНИ: первые 80% — обучение, последние 20% — тест
split = int(len(df) * 0.8)
train = df.iloc[:split]   # прошлое
test = df.iloc[split:]    # будущее

# НИКОГДА не используй train_test_split(shuffle=True) для рядов!

**Внимание  ⚠️**

Случайное перемешивание (обычный train_test_split) для временных рядов — грубая ошибка и скрытая утечка: модель «увидит» будущие точки при обучении и покажет фантастическую, но ложную точность. Всегда режь ряд по времени: обучение — раньше, тест — позже.

## Baseline: скользящее среднее

**Аналогия: завтра будет как в среднем недавно  💡**

Простейший разумный прогноз: «завтрашнее значение примерно равно среднему за последние несколько дней». Это скользящее среднее (moving average). Оно сглаживает случайные скачки и часто на удивление неплохо предсказывает. Как baseline из главы 6, оно задаёт планку: сложная модель должна обгонять этот простой прогноз, иначе она не нужна.

In [ ]:
import matplotlib.pyplot as plt

# Скользящее среднее за 7 дней как простой прогноз
df["прогноз"] = df["value"].rolling(window=7).mean()

# Смотрим, как сглаженная линия следует за данными
df[["value", "прогноз"]].plot()
plt.title("Данные и скользящее среднее (baseline)")
plt.show()

## Специальные признаки времени

Как и в feature engineering (глава 26), из времени извлекают полезные признаки, но здесь особенно важны признаки «прошлого»: значение вчера, значение неделю назад (лаги), скользящее среднее, день недели, сезон. Именно они позволяют обычным моделям вроде случайного леса работать с временем.

In [ ]:
# Лаги — значения из прошлого как признаки
df["вчера"] = df["value"].shift(1)
df["неделю_назад"] = df["value"].shift(7)
df["среднее_7"] = df["value"].rolling(7).mean().shift(1)
# shift обязателен: берём ТОЛЬКО прошлое, не текущий день

## Готовые инструменты

Для серьёзных прогнозов есть специальные инструменты. Prophet от Meta — прост в использовании, сам учитывает сезонность и праздники. ARIMA — классический статистический метод. Но и обычный случайный лес на лаговых признаках часто работает отлично — начни с него и скользящего среднего как baseline.

## Локальный пример: курс сомони НБТ

Представь прогноз курса доллара к сомони по данным Национального банка Таджикистана. Правила те же: обучение на прошлых курсах, тест на последних неделях (по времени!), baseline — скользящее среднее за неделю, признаки — лаги (курс вчера, неделю назад). Важно помнить: финансовые ряды крайне трудно предсказуемы, и хороший прогноз здесь — скромное улучшение над baseline, а не магическое угадывание. Это честный урок о пределах ML.

## Задания

Базовый уровень

Возьми любой временной ряд (погода, продажи, посещения). Раздели его ПО ВРЕМЕНИ на train/test. Построй скользящее среднее за 7 дней как baseline и нарисуй его поверх данных. Насколько хорошо оно следует за реальностью?

Со звёздочкой  ⭐

Создай лаговые признаки (вчера, неделю назад, скользящее среднее) и обучи случайный лес прогнозировать следующее значение. Сравни с baseline (скользящее среднее) на тесте по MAE. Обгоняет ли модель простой baseline? Не забудь: разбиение по времени, shift для лагов.

Вопросы для обсуждения  💬

Почему для временных рядов нельзя случайно перемешивать данные?

Когда скользящее среднее — уже достаточно хороший прогноз, а когда нет?

Почему прогноз курса валют — одна из самых сложных задач ML?

**Проверь себя  ℹ️**

1. Чем разбиение временного ряда отличается от обычного train/test?

2. Что такое скользящее среднее и зачем оно как baseline?

3. Что такое лаговые признаки?